# Cleaning Data in Python

## Chapter 1 - Common data problems

### Data type constraints
### Data range constraints
### Uniqueness constraints

In [ ]:
import pandas as pd

ride_sharing = pd.read_csv(r'D:\Data Engineer\Python\MKD_Classes\Base\Datasets\ride_sharing_new.csv')

print(ride_sharing.info())


In [ ]:
print(ride_sharing['user_type'].describe())
print(ride_sharing.head(2))

In [ ]:
ride_sharing['user_type_cat'] = ride_sharing['user_type'].astype('category')

In [ ]:
assert ride_sharing['user_type_cat'].dtype == 'category'

In [ ]:
ride_sharing['user_type_cat'].describe()

In [ ]:
ride_sharing['duration_trim'] = ride_sharing['duration'].str.strip('minutes')
#converts the duration column to a string type and removes the 'minutes' string from the values.

In [ ]:
ride_sharing['duration_time'] = ride_sharing['duration_trim'].astype('int')
#converts the duration column to an integer type after stripping the 'minutes' string from the values.

In [ ]:
assert ride_sharing['duration_time'].dtype == 'int'
#assertion checks that the 'duration_time' column is of integer type.

In [ ]:
print(ride_sharing[['duration', 'duration_trim', 'duration_time']].head(2))
print(round(sum(ride_sharing['duration_time']/(ride_sharing['duration_time'].count()))))

### Data Range constraints

In [ ]:
# Convert tire_sizes to integer
ride_sharing['tire_sizes'] = ride_sharing['tire_sizes'].astype('int')

# Set all values above 27 to 27
#Loc selects rows, applies a filter, and selects given columns
ride_sharing.loc[ride_sharing['tire_sizes'] > 27, ['tire_sizes']] = 27

# Reconvert tire_sizes back to categorical
ride_sharing['tire_sizes'] = ride_sharing['tire_sizes'].astype('category')

# Print tire size description
print(ride_sharing['tire_sizes'].info())

In [ ]:
# Convert ride_date to date
ride_sharing['ride_dt'] = pd.to_datetime(ride_sharing['ride_date']).dt.date

# Save today's date
today = dt.date.today()

# Set all in the future to today's date
ride_sharing.loc[ride_sharing['ride_dt'] > today, 'ride_dt'] = today

# Print maximum of ride_dt column
print(ride_sharing['ride_dt'].max())

### Uniqueness constraints

Dealing with duplicate values

#Get duplicates across all columns - .duplicated()

duplicates = df.duplicated()
<returns boolean>

//
#check duplicates in df format:
df[duplicates]


### Using the .duplicated() method:

subset = list of column names to check for duplication
keep = whether to keep first, last or all duplicate values (first, last, false)

- Python
#column names to check for duplication
column_names = ['first_name','last_name', 'address']

#check for duplicates on specific columns and keep all:
duplicates = height_weight.duplicated(subset=column_names, keep = False)

#output duplicate values
height_weight[duplicates].sort_values(by = 'first_name')

#.sort_values() method:
.sort_values(by = 'first_name', ascending = False)

#sorting by multiple columns: 
.sort_values(by = ['first_name','last_name'] ascending = ['False','True'])

### the .drop_duplicates() method:


subset = list of column names to check for duplication
keep = first, last, False
inplace = Drop duplicated rows directly inside DataFrame without creating a new object

- Python
#drop duplicates
df_height_weight.drop_duplicates(inplace = True)

### The .groupby() and .agg() methods

- Python
#Group by column names and produce statistical summaries
column_names = ['first_name','last_name', 'address']
#create a dict
summaries = {'height':'max', 'weight':'mean'}
df_height_weight = df+height_weight.groupby(by = column_names).agg(summaries).reset_index()

- df.groupby(by = column_names)
This takes your dataset and splits it into distinct buckets based on the columns listed in the column_names variable. For example, if column_names = ['Gender', 'Age_Group'], Pandas will create a separate underlying group for every unique combination of Gender and Age Group.

- .agg(summaries)
Short for "aggregate," this method applies calculations to the groups created in the previous step. The summaries variable is usually a dictionary dictating exactly which math functions to apply to which columns. For example, if summaries = {'Height': 'mean', 'Weight': 'max'}, it will calculate the average height and the maximum weight for each group.

- .reset_index()
When Pandas performs a .groupby(), the columns you grouped by (e.g., Gender and Age Group) are automatically turned into the new "index" (row labels) of the resulting table. .reset_index() pops those index labels back out into standard columns and assigns a clean, normal index (0, 1, 2...) to the new DataFrame.



In [ ]:
#find duplicates exercise:

# Find duplicates:
duplicates = ride_sharing.duplicated(subset = 'ride_id', keep = False)

# Sort your duplicated rides
duplicated_rides = ride_sharing[duplicates].sort_values('ride_id', ascending = True)

# Print relevant columns of duplicated_rides
print(duplicated_rides[['ride_id','duration','user_birth_year']])

In [ ]:
#Treating duplicates exercise:
# Drop complete duplicates from ride_sharing
ride_dup = ride_sharing.drop_duplicates()

# Create statistics dictionary for aggregation function
statistics = {'user_birth_year': 'min', 'duration': 'mean'}

# Group by ride_id and compute new statistics
ride_unique = ride_dup.groupby('ride_id').agg(statistics).reset_index()

# Find duplicated values again
duplicates = ride_unique.duplicated(subset = 'ride_id', keep = False)
duplicated_rides = ride_unique[duplicates == True]

# Assert duplicates are processed
assert duplicated_rides.shape[0] == 0

### Exercise breakdown:

1. Removing Exact Duplicates
Python
ride_dup = ride_sharing.drop_duplicates()
.drop_duplicates(): This method scans the ride_sharing DataFrame and removes any rows that are 100% identical across every single column. The cleaned dataset is saved as ride_dup.

2. Defining Aggregation Rules
Python
statistics = {'user_birth_year': 'min', 'duration': 'mean'}
This creates a standard Python dictionary. It dictates how Pandas should handle data when it merges rows together in the next step. It tells Pandas: "When you combine rows, keep the minimum (min) value found in user_birth_year, and calculate the average (mean) of the duration values."

3. Combining Partial Duplicates
Python
ride_unique = ride_dup.groupby('ride_id').agg(statistics).reset_index()
.groupby('ride_id'): This groups the data so that all rows sharing the exact same ride_id are bucketed together.
.agg(statistics): This applies the dictionary you created above. If a ride_id had two rows, it collapses them into one by taking the oldest birth year and the average duration of the two rows.
.reset_index(): Because .groupby() automatically turns the grouped column (ride_id) into the row index, this function pops ride_id back out into a standard column and gives the DataFrame a normal numbered index (0, 1, 2...).

4. Checking for Remaining Duplicates
Python
duplicates = ride_unique.duplicated(subset = 'ride_id', keep = False)
duplicated_rides = ride_unique[duplicates == True]
.duplicated(subset = 'ride_id', keep = False): This checks the new DataFrame to see if any duplicate ride_ids survived the grouping process. subset tells it to only look at the ride_id column. keep = False tells Pandas to mark every instance of a duplicate as True (rather than keeping the first one it sees as a valid original). It returns a list of True/False values.
ride_unique[duplicates == True]: This uses the True/False list as a filter to create a new DataFrame (duplicated_rides) containing only the rows that failed the duplicate check.

5. Running a Sanity Check
Python
assert duplicated_rides.shape[0] == 0
.shape[0]: The .shape attribute returns the dimensions of a DataFrame as (rows, columns). Adding [0] specifically extracts the number of rows.
assert: This is a built-in Python testing keyword. It evaluates the statement next to it (duplicated_rides.shape[0] == 0). If the statement is true (meaning there are 0 rows of duplicates), the script continues silently. If it is false, Python will crash the script and throw an AssertionError, alerting you that your data cleaning failed.

## Chapter 2 - Text and Categorical Data

### Membership constraints
### Categorical variables
### Cleaning text data

Categories = variables that represents a finite set of categories

A good practice is to keep a log of all possible categorical data, this will make dealing with inconsistencies easier.

We can use an anti join using the original df and the log of categorical data to exclude all possible and valid categories, keeping only the ones we want to remove

inconsistent_categories = set(original_df['column']).difference(categories['log_categories_accepted'])
#anti join, removing all acceptable categories from original df, leaving only the inconsistent ones and assigning those to an object. 
#difference()

inconsistent_rows = original_df['column'].isin(inconsistent_categories)
#creating a df with the rows that contains inconsistent categories
#isin() <= returns a boolean list, acts as a filter, keeping only True values

{
    Python
import pandas as pd

df = pd.DataFrame({'status': ['pending', 'active', 'closed', 'active']})
target_statuses = ['active', 'pending']

#Creates a boolean mask: [True, True, False, True]
mask = df['status'].isin(target_statuses)

#Filters the DataFrame to only keep 'active' and 'pending' rows
filtered_df = df[mask]

}


### Dropping inconsistent categoris with the ~ symbol

#drop inconsistent categories

consistent_data = original_df[~inconsistent_rows]
the ~ representes 'except', so we are assigning the original DF, except inconsistent rows, to a consistent_data df.

### Finiding inconsistency exercise
- Python 

#Print categories DataFrame
print(categories)

#Print unique values of survey columns in airlines
print('Cleanliness: ', airlines['cleanliness'].unique(), "\n")
print('Safety: ', airlines['safety'].unique(), "\n")
print('Satisfaction: ', airlines['satisfaction'].unique(), "\n")

.unique() method <= works like DISTINCT from SQL, returning unique results only

### Finding CONsistency

#Find the cleanliness category in airlines not in categories
cat_clean = set(airlines['cleanliness']).difference(categories['cleanliness'])

#Find rows with that category
cat_clean_rows = airlines['cleanliness'].isin(cat_clean)

#Print rows with inconsistent category
print(airlines[cat_clean_rows])

### 2.1 Categorical Variables

- 1 Value Inconsistency
inconsistent fields
trailing white spaces

- 2 Collapsing too many categories to few
creating new groups
mapping groups to new ones

- 3 Making sure it has the right type


### Value Consistency

#Capitalize
.str.upper()

#Lowercase
.str.lower()

#Strip all leading an trailing spaces
.str.strip()

### Collapsing data into categories

#using qcut()

import pandas as pd
group = ['1', '2', '3']
original_df['new_column'] = pd.qcut(original_df['base_column'], q = 3, labels = group)

#using cut()
#create category ranges and names

ranges = [0, 200000, 500000, np.inf]
group = ['0-200k', '200k-500k', '500K+']

#create income group:
original_df['new_column'] = pd.cut(original_df['base_column'], bins = ranges, labels = group_names)

#Using .replace() + unique
#Create mapping dictionary and replace

mapping_dict = {key:value} dict

original_df['new_column'] = original_df['base_column'].replace(mapping_dict)
#replace returns a Series, not Categorical, so add .astype('category') in the chain


### Inconsistent categories - exercise 1

#Print unique values of both columns
print(airlines['dest_region'].unique())
print(airlines['dest_size'].unique())

#Lower dest_region column and then replace "eur" with "europe"
airlines['dest_region'] = airlines['dest_region'].str.lower() 
airlines['dest_region'] = airlines['dest_region'].replace({'eur':'europe'})

#Remove white spaces from `dest_size`
airlines['dest_size'] = airlines['dest_size'].str.strip()

#Verify changes have been effected
print(airlines['dest_region'].unique())
print(airlines['dest_size'].unique())

### Remapping categories - exercise 2

#Create ranges for categories
label_ranges = [0, 60, 180, np.inf]
label_names = ['short', 'medium', 'long']

#Create wait_type column
airlines['wait_type'] = pd.cut(airlines['wait_min'], bins = label_ranges, 
                                labels = label_names)

#Create mappings and replace
mappings = {'Monday':'weekday', 'Tuesday':'weekday', 'Wednesday': 'weekday', 
            'Thursday': 'weekday', 'Friday': 'weekday', 
            'Saturday': 'weekend', 'Sunday': 'weekend'}

airlines['day_week'] = airlines['day'].replace(mappings)

### Cleaning text data and regex

### Removing titles and taking names - Exercise 1

- Python
#Replace "Dr." with empty string ""
airlines['full_name'] = airlines['full_name'].str.replace("Dr.","")

#Replace "Mr." with empty string ""
airlines['full_name'] = airlines['full_name'].str.replace("Mr.","")

#Replace "Miss" with empty string ""
airlines['full_name'] = airlines['full_name'].str.replace("Miss","")

#Replace "Ms." with empty string ""
airlines['full_name'] = airlines['full_name'].str.replace("Ms.","")

#Assert that full_name has no honorifics
assert airlines['full_name'].str.contains('Ms.|Mr.|Miss|Dr.').any() == False

### Keeping it descriptive - Exercise 2

- Python - usning mapping and returning the full df filtered

#Store length of each row in survey_response column
resp_length = airlines['survey_response'].str.len()

#Find rows in airlines where resp_length > 40
airlines_survey = airlines[resp_length > 40]

#Assert minimum survey_response length is > 40
assert airlines_survey['survey_response'].str.len().min() > 40

#Print new survey_response column
print(airlines_survey['survey_response'])

- Python - using .loc() and returning a series only:"

#Store length of each row in survey_response column
resp_length = airlines['survey_response'].str.len()

#Find rows in airlines where resp_length > 40
airlines_survey = airlines.loc[resp_length > 40, 'survey_response']

#Assert minimum survey_response length is > 40
assert airlines_survey.str.len().min() > 40

#Print new survey_response column
print(airlines_survey)

## Chapter 3 - Advanced data problems 

### Uniformity
### Cross field validation
### Completeness

## Uniformity

### Uniform currencies exercise

- python 

#Find values of acct_cur that are equal to 'euro'
acct_eu = banking['acct_cur'] == 'euro'

#Convert acct_amount where it is in euro to dollars
banking.loc[acct_eu, 'acct_amount'] = banking.loc[acct_eu, 'acct_amount'] * 1.1

#Unify acct_cur column by changing 'euro' values to 'dollar'
banking.loc[acct_eu, 'acct_cur'] = 'dollar'

#Assert that only dollar currency remains
assert banking['acct_cur'].unique() == 'dollar'

- python

#Print the header of account_opend
print(banking['account_opened'].head())

#Convert account_opened to datetime
banking['account_opened'] = pd.to_datetime(banking['account_opened'],
                                           # Return missing value for error
                                           errors = 'coerce') 

#Get year of account opened
banking['acct_year'] = banking['account_opened'].dt.strftime('%Y')

#Print acct_year
print(banking['acct_year'])

## Cross Field validation

### Data integrity - Exercise 1

#Store today's date and find ages
today = dt.date.today()
ages_manual = today.year - banking['birth_date'].dt.year

#Find rows where age column == ages_manual
age_equ = banking['age'] == ages_manual

#Store consistent and inconsistent data
consistent_ages = banking[age_equ]
inconsistent_ages = banking[~age_equ]

#Store consistent and inconsistent data
print("Number of inconsistent ages: ", inconsistent_ages.shape[0])

## Completeness - Missing Data issues

#return missing values:
df.isna() 
returns a series of booleans, True if missing (if NaN)

#Package for visualizing missing data
Missingno
import missingno as msno

- 1 #visualize missingness
msno.matrix(df)
plt.show

- 2 #isolate missing and complete values
missing = df[df['column_missing_values'].isna()]
complete = df[~df['column_missing_values'].isna()]

#.isna() returns a map/filter, with boolean values, and is being applied inside DF to return it filtered with only true values(for missing) or false values(for complete)

- 3 #use the .describe() method to analyze the two
complete.describe()
missing.describe()

- 4 #visualize the missing df with msno.matrix
sorted_df = df.sort_values(by = 'column_missing_values')
msno.matrix(sorted_df)
plt.show

### How to deal with missing data?
- drop missing data
- impute with statistical (mean, median, mode)
- imputing using an algorithmic approach
- impute with machine learning models

- 5 #drop missing values:
df_dropped = df.dropna(subset = ['column_missing_values'])

- 6 #replace with statistical measures
col_mean = df['col_mis_val'].mean()
df_imputed = df.fillna({'col_ms_val':col_mean})



### Missing investors - Exercise:

- Python

#Print number of missing values in banking
print(banking.isna().sum())

#Visualize missingness matrix
msno.matrix(banking)
plt.show()

#Isolate missing and non missing values of inv_amount
missing_investors = banking[banking['inv_amount'].isna()]
investors = banking[~banking['inv_amount'].isna()]

#Sort banking by age and visualize
banking_sorted = banking.sort_values(by = 'age')
msno.matrix(banking_sorted)
plt.show()

### Follow the money - Exercise:

#Drop missing values of cust_id
banking_fullid = banking.dropna(subset = ['cust_id'])

#Compute estimated acct_amount
acct_imp = banking_fullid['inv_amount'] * 5

#Impute missing acct_amount with corresponding acct_imp
banking_imputed = banking_fullid.fillna({'acct_amount': acct_imp})

#Print number of missing values
print(banking_imputed.isna().sum())

# Chapter 4 - Record linkage

Minimum edit distance = least possible amount of steps needed to transition from one string to another

There are lots of minimum edit distance algorithms
the lower the edit distance, the closer two different words are

possible packages: nltk, thefuzz, textdistance

We'll use levenshtein using thefuzz package

from thefuzz import fuzz
fuzz.WRatio('Reeding', 'Reading') returns 86

score from 0 to 100 


### Comparison with arrays
string = "string"
choices = pd.Series([array])

process.extract(string, chiices, limit=)

### Collapsing categories with string matching

### Exercise - establishing the cutoff point for the similarity score:

#Import process from thefuzz
from thefuzz import process

#Store the unique values of cuisine_type in unique_types
unique_types = restaurants['cuisine_type'].unique()

#Calculate similarity of 'asian' to all values of unique_types
print(process.extract('asian', unique_types, limit = len(unique_types)))

#Calculate similarity of 'american' to all values of unique_types
print(process.extract('american', unique_types, limit = len(unique_types)))

#calculate similarity of 'italian' to all values of unique_types
print(process.extract('italian', unique_types, limit = len(unique_types)))



In [15]:
import pandas as pd
from thefuzz import process

restaurants = pd.read_csv(r'D:\Data Engineer\Python\MKD_Classes\Base\Datasets\restaurants_L2.csv')


#Store the unique values of cuisine_type in unique_types
unique_types = restaurants['type'].unique()
print(unique_types)

#Calculate similarity of 'asian' to all values of unique_types
print(process.extract('asian', unique_types, limit = len(unique_types)))

#Calculate similarity of 'american' to all values of unique_types
print(process.extract('american', unique_types, limit = len(unique_types)))

#calculate similarity of 'italian' to all values of unique_types
print(process.extract('italian', unique_types, limit = len(unique_types)))

#set cutoff score of 80
#Remapping categories II

# Create a list of matches, comparing 'italian' with the cuisine_type column
matches = process.extract('italian', restaurants['type'], limit=len(restaurants.type))

# Iterate through the list of matches to italian
for match in matches:
  # Check whether the similarity score is greater than or equal to 80
  if match[1] >= 80:
    # Select all rows where the cuisine_type is spelled this way, and set them to the correct cuisine
    restaurants.loc[restaurants['type'] == match[0],'type'] = 'italian'



<ArrowStringArray>
[    'american',        'asian',      'italian',    'coffeebar',
      'mexican', 'southwestern',  'steakhouses',     'southern',
        'cajun']
Length: 9, dtype: str
[('asian', 100), ('italian', 67), ('american', 62), ('mexican', 50), ('steakhouses', 40), ('cajun', 40), ('southwestern', 36), ('southern', 31), ('coffeebar', 26)]
[('american', 100), ('mexican', 80), ('cajun', 68), ('asian', 62), ('italian', 53), ('southwestern', 49), ('southern', 38), ('coffeebar', 24), ('steakhouses', 21)]
[('italian', 100), ('asian', 67), ('american', 53), ('mexican', 43), ('cajun', 33), ('southwestern', 33), ('steakhouses', 33), ('southern', 27), ('coffeebar', 12)]


In [16]:
#Remapping categories III
#Finally, you'll adapt your code to work with every restaurant type in categories.

categories = ['asian', 'american', 'italian', 'southern']

for cuisine in categories:
    matches = process.extract(cuisine, restaurants['type'], limit=len(restaurants.type))
    print(matches)
    for match in matches:
        if match[1] >= 80:
            restaurants.loc[restaurants['type'] == match[0],'type'] = cuisine

print(restaurants['type'].unique())

[('asian', 100, 5), ('asian', 100, 7), ('asian', 100, 12), ('asian', 100, 18), ('asian', 100, 31), ('asian', 100, 32), ('asian', 100, 38), ('asian', 100, 46), ('asian', 100, 53), ('asian', 100, 54), ('asian', 100, 67), ('asian', 100, 76), ('asian', 100, 77), ('asian', 100, 79), ('asian', 100, 91), ('asian', 100, 95), ('asian', 100, 96), ('asian', 100, 98), ('asian', 100, 101), ('asian', 100, 111), ('asian', 100, 116), ('asian', 100, 129), ('asian', 100, 130), ('asian', 100, 142), ('asian', 100, 144), ('asian', 100, 157), ('asian', 100, 162), ('asian', 100, 171), ('asian', 100, 173), ('asian', 100, 174), ('asian', 100, 179), ('asian', 100, 189), ('asian', 100, 193), ('asian', 100, 196), ('asian', 100, 199), ('asian', 100, 200), ('asian', 100, 205), ('asian', 100, 206), ('asian', 100, 210), ('asian', 100, 214), ('asian', 100, 215), ('asian', 100, 226), ('asian', 100, 229), ('asian', 100, 231), ('asian', 100, 239), ('asian', 100, 248), ('asian', 100, 257), ('asian', 100, 258), ('asian', 1

### Record linkage

Act of linking data from different sources.
Clean 2 or more dataframes, generate pairs of potential matches
score these pairs according to string similarity(thefuzz)
link them

all these steps can be achieved with the recordlinkage package

import recordlinkage

#create indexing object
indexer = recordlinkage.Index()

this object will be used to generate pairs from our dataset

#Generate pairs "blocked" on state column
#we block the pairs because pairing every record from a table with all other records from the other tables would be unpractical and take too much time. Blocking solves this by using a column to create pairs that match this column

indexer.block('state')
pairs = indexer.index(census_A, census_B)

the resulting "pairs"object is a pandas multi index object, with pairs of row indices from both dataframes

Comparing the DataFrames:

#generate the pairs
pairs = indexer.index(census_A, census_B)

#create a compare object
compare_cl = recordlinkage.Compare()

#find exact matches for pairs of date_of_birth and state

compare_cl.exact('date_of_birth', 'date_of_birth', label='date_of_birth')
compare_cl.exact('state', 'state', label='state')

#find SIMILAR matches for pairs of surname and address_1 - using string similarity
compare_cl.string('surname', 'surname', threshold=0.85, label='surname')
compare_cl.string('address_1', 'address_1', threshold=0.85, label='address_1')

#find matches = compute the matches with the compute function
potential_matches = compare_cl.compute(pairs, census_A, census_B)

print(potential_matches)
returns a df with the columns being compared, with values = 1 for a match, and =0 for a not match


#finding only pairs we want
potential_matches[potential_matches.sum(axis=1) => 2]

in this case, since we want two columns matching, we can filter for sum(axis=1) => 2, so it means that we have 2 columns with "match" value of 1 (1+1=>2)


Exercise - Similar restaurants
#Create a comparison object
comp_cl = recordlinkage.Compare()

#Find exact matches on city, cuisine_types - 
comp_cl.exact('city', 'city', label='city')
comp_cl.exact('cuisine_type', 'cuisine_type', label='cuisine_type')

#Find similar matches of rest_name
comp_cl.string('rest_name', 'rest_name', label='name', threshold = 0.8) 

#Get potential matches and print
potential_matches = comp_cl.compute(pairs, restaurants, restaurants_new)
print(potential_matches)